<a href="https://colab.research.google.com/github/a4kashhh/MaxBetweennessGNN/blob/main/HopesNeverDie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import networkx as nx
import numpy as np
import random
import torch
#creating BA network + their properities
G = nx.barabasi_albert_graph(n=200,m=4,seed=42)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 200
Edges: 784


In [19]:
#phir se we are doing some properties stuffu + slicing
degree = dict(G.degree())
clustering = nx.clustering(G)
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
pagerank = nx.pagerank(G)
features = []
for node in G.nodes():
    features.append([degree[node],clustering[node],betweenness[node],closeness[node],pagerank[node]])
x = torch.tensor(features,dtype=torch.float)
print(x.shape)
print(x[:5])

torch.Size([200, 5])
tensor([[6.6000e+01, 5.5944e-02, 2.5405e-01, 5.8876e-01, 3.7010e-02],
        [1.3000e+01, 1.0256e-01, 1.8824e-02, 4.4420e-01, 7.8242e-03],
        [5.0000e+00, 5.0000e-01, 3.7574e-04, 4.0202e-01, 3.2733e-03],
        [2.1000e+01, 1.1429e-01, 3.7807e-02, 4.8537e-01, 1.1881e-02],
        [8.0000e+00, 1.7857e-01, 7.8297e-03, 4.1286e-01, 5.0950e-03]])


In [20]:
degree, clustering, betweenness, pagerank

({0: 66,
  1: 13,
  2: 5,
  3: 21,
  4: 8,
  5: 30,
  6: 28,
  7: 38,
  8: 40,
  9: 20,
  10: 21,
  11: 10,
  12: 19,
  13: 13,
  14: 17,
  15: 11,
  16: 20,
  17: 18,
  18: 23,
  19: 13,
  20: 12,
  21: 21,
  22: 9,
  23: 22,
  24: 17,
  25: 17,
  26: 8,
  27: 10,
  28: 11,
  29: 16,
  30: 9,
  31: 7,
  32: 11,
  33: 7,
  34: 17,
  35: 7,
  36: 8,
  37: 16,
  38: 13,
  39: 6,
  40: 7,
  41: 11,
  42: 9,
  43: 5,
  44: 5,
  45: 8,
  46: 8,
  47: 5,
  48: 7,
  49: 5,
  50: 12,
  51: 4,
  52: 7,
  53: 7,
  54: 7,
  55: 10,
  56: 9,
  57: 8,
  58: 9,
  59: 12,
  60: 6,
  61: 8,
  62: 8,
  63: 4,
  64: 4,
  65: 6,
  66: 7,
  67: 5,
  68: 9,
  69: 5,
  70: 7,
  71: 6,
  72: 8,
  73: 8,
  74: 14,
  75: 7,
  76: 6,
  77: 5,
  78: 5,
  79: 4,
  80: 4,
  81: 6,
  82: 6,
  83: 7,
  84: 4,
  85: 12,
  86: 5,
  87: 7,
  88: 7,
  89: 11,
  90: 7,
  91: 6,
  92: 5,
  93: 6,
  94: 6,
  95: 4,
  96: 5,
  97: 4,
  98: 6,
  99: 6,
  100: 4,
  101: 4,
  102: 5,
  103: 7,
  104: 8,
  105: 6,
  106: 4,
  1

In [21]:
#misssiing edges findinggggggss
def generate_missing_edges(G):
    nodes = list(G.nodes())
    missing = []
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if not G.has_edge(i, j):
                missing.append((i, j))
    return missing
missing_edges = generate_missing_edges(G)
print("Total Missing Links:",len(missing_edges))

Total Missing Links: 19116


In [22]:
#how much bwtnes centrality is affected by new edge
#**GROUND TRUTHHH**
def edge_impact(G,edge):
    bc_original = nx.betweenness_centrality(G)
    max_original = max(bc_original.values())
    G_new = G.copy()
    G_new.add_edge(edge[0],edge[1])
    bc_new = nx.betweenness_centrality(G_new)
    max_new = max(bc_new.values())
    return (max_original -max_new)

In [23]:
# randomly select at most 500 missing edges (reduces computation because BC calculation is expensive)
missing_edges = random.sample(missing_edges,min(500,len(missing_edges)))
scores = []
# check every sampled missing edge
for edge in missing_edges:
    # calculate how much maximum betweenness decreases
    # after adding this edge
    reduction = edge_impact(G,edge)
    scores.append(reduction)
    #transforming it into tensor cause we are dealing with pytorchh :)
scores = torch.tensor(scores,dtype=torch.float)
# display first 10 scores
# each score = BC reduction produced by that edge
print(scores[:10])

tensor([4.0859e-04, 3.9007e-06, 1.5249e-04, 9.0045e-05, 2.3643e-04, 4.9106e-04,
        2.9203e-04, 1.5528e-03, 4.8819e-04, 6.7150e-05])


In [24]:
#normalisation :) z score (x - xbar /  sd)
scores = (scores-scores.mean())/(scores.std())
print(scores.min())
print(scores.max())

tensor(-10.7915)
tensor(4.0748)


In [25]:
#cutoff //
# Create class labels
# 1 = important edge (top 10%)
# 0 = normal edge (remaining 90%)
threshold = np.percentile(scores.numpy(),90)
labels = torch.tensor([1 if s >= threshold else 0 for s in scores],dtype=torch.long)
# Count how many 0s and 1s were created
print(labels.unique(return_counts=True))

(tensor([0, 1]), tensor([450,  50]))


In [26]:
#converting missing_edges from a Python list into a PyTorch tensor.
candidate_edges = torch.tensor(missing_edges,dtype=torch.long).t()
print(candidate_edges.shape)

torch.Size([2, 500])


In [27]:
#Triplet Margin Loss
def generate_triplets(scores):
    idx = torch.argsort(scores,descending=True) #Sorting edges score ke basis pe
    n = len(idx)#Number of edges.
    top = idx[:n//3] #take third best
    middle = idx[n//3:2*n//3] #middle third
    bottom = idx[2*n//3:] #bottom third
    triplets = []
    for i in range( #Find the smallest list length and loop that many times.
        min(len(top),len(middle),len(bottom))):
        triplets.append( #Add a new element to the triplets list.
            (top[i],middle[i],bottom[i]))
    return triplets #wapas se us function pe
triplets = generate_triplets(scores)
print(len(triplets))

166


In [28]:
!pip torch_geometric.nn
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_networkx

ERROR: unknown command "torch_geometric.nn"


In [29]:
data = from_networkx(G) #conbert NetworkX graph into a PyTorch Geometric graph object
edge_index = data.edge_index #Extract graph connections.
print(edge_index.shape) #Print size of edge index

torch.Size([2, 1568])


In [30]:
class GraphSAGEEncoder(nn.Module):
    def __init__( self, input_dim, hidden_dim=32):  #Constructor // input_dim - Number of input features per node.
        super().__init__()
        self.conv1 = SAGEConv(input_dim,hidden_dim)
        self.conv2 = SAGEConv(hidden_dim,hidden_dim)
    def forward(self,x,edge_index):
        x = self.conv1(x,edge_index) #Aggreg neighbor information.
        x = F.relu(x)
        x = self.conv2(x,edge_index) #Perform another round  message passing.
        return x

In [31]:
class MLPClassifier(nn.Module):
    def __init__(self): #Constructor
        super().__init__()
        self.fc1 = nn.Linear(64,32)
        self.fc2 = nn.Linear(32,2) #Binary classification: // Class 0 = Not Important Edge // Class 1 = Important Edge
    def forward(self,edge_emb):
        x = F.relu(self.fc1(edge_emb))
        return self.fc2(x)

In [32]:
class MLPRanker(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64,32)
        self.fc2 = nn.Linear(32,1) #one importance score per edge groupp
    def forward(self,edge_emb):
        x = F.relu(self.fc1(edge_emb))
        return self.fc2(x) #ranking score.

In [33]:
class BetweennessLinkPredictor(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.encoder = (GraphSAGEEncoder(input_dim))#Create GraphSAGE network.
        self.classifier = (MLPClassifier())#Create MPL 1 for CLASSIFICATION
        self.ranker = (MLPRanker()) #creates MLP 2 i.e for ranking
    def forward(self,x,edge_index,candidate_edges): #x - node features // edge_index - graph connection // candidate_edges - Missing edges to evaluate
        node_emb = self.encoder(x,edge_index) # calls  GraphSAGE.
        src = candidate_edges[0] #take source nodes
        dst = candidate_edges[1] #take destination nodes
        edge_emb = torch.cat([node_emb[src],node_emb[dst]],dim=1) #Create an edge embedding from two node embeddings.
        class_logits = (self.classifier(edge_emb)) #Send edge embedding MLP1 Classifier
        rank_scores = (self.ranker(edge_emb)) #Send edge embedding MLP2 RANKER
        return (class_logits,rank_scores)

In [34]:
def triplet_loss(rank_scores,triplets,margin=1.0): #Inputs // rank_scores - Scores predicted by MLP Ranker //. triplets // margin - Minimum separation required.
    loss = 0 # total loss
    for a,b,c in triplets: #loop
        loss += F.relu(rank_scores[b]-rank_scores[a]+margin) #loss calcualtion (Is Good Edge ranked higher than Medium Edge? )
        loss += F.relu(rank_scores[c]-rank_scores[b]+margin)  #loss calcualtion(Is Medium Edge ranked higher than Bad Edge?)
    return (loss /len(triplets)) #averageloss




In [35]:
def total_loss(class_logits,labels,rank_scores,triplets):  #total_loss
  classification_loss = (F.cross_entropy(class_logits,labels)) #Calculate classifier erro
  ranking_loss = (triplet_loss(rank_scores,triplets)) #Calculate ranking error.
  return (classification_loss+ranking_loss) #Add both losi

In [36]:
model = (BetweennessLinkPredictor(input_dim=x.shape[1])) # Create model
optimizer = torch.optim.Adam(model.parameters(),lr=0.001) # Create model
for epoch in range(100): # Training loop
    class_logits, rank_scores = model(x,edge_index,candidate_edges)     #Forward pass
    loss = total_loss(class_logits,labels,rank_scores,triplets) #Compute loss
    optimizer.zero_grad()
    loss.backward() #backwarrdd propagration
    optimizer.step() #Compute loss
    if epoch % 10 == 0:
        print(epoch,loss.item())

0 2.7942776679992676
10 2.058907985687256
20 1.9521440267562866
30 1.8616397380828857
40 1.749392032623291
50 1.6469084024429321
60 1.5617566108703613
70 1.4881045818328857
80 1.417060375213623
90 1.3492680788040161
